# Raw PDF OCR Processing Notebook

本 notebook 用于处理原始 PDF 论文，并与 Open RAG Benchmark 已提供的 JSON 语料进行对比。

主要目标：
1. 读取 `pdf_urls.json`；
2. 下载 50 篇 raw PDFs；
3. 使用 OCR / markdown 提取方式得到论文内容；
4. 将 markdown 进一步切成 section；
5. 保存 OCR 解析结果；
6. 与官方 JSON 解析结果做基础统计对比。

**注意：**
- 当前版本使用 `pymupdf4llm.to_markdown(...)` 作为 PDF → markdown 的提取入口；
- 若后续需要更纯粹的 PaddleOCR / PP-Structure 实现，只需替换提取函数；
- 所有输出统一写入：`project/data/rawpdf/`。

## 0. 依赖说明

建议安装：
```bash
pip install requests tqdm pymupdf pymupdf4llm
# 可选：pip install paddleocr paddlepaddle
```

In [ ]:
from __future__ import annotations

import json
import re
from pathlib import Path

import requests
from tqdm import tqdm

# ── 路径推断：优先用 notebook 文件自身位置，回退到 cwd ──────────────────────
_nb_file = globals().get('__vsc_ipynb_file__')
if _nb_file:
    PROJECT_ROOT = Path(_nb_file).resolve().parent.parent
else:
    _cwd = Path.cwd()
    if _cwd.name == 'code':
        PROJECT_ROOT = _cwd.parent
    elif (_cwd / 'project').exists():
        PROJECT_ROOT = _cwd / 'project'
    elif _cwd.name == 'project':
        PROJECT_ROOT = _cwd
    else:
        PROJECT_ROOT = _cwd

DATA_ROOT       = PROJECT_ROOT / 'data' / 'open_ragbench' / 'pdf' / 'arxiv'
PDF_URLS_PATH   = DATA_ROOT / 'pdf_urls.json'
JSON_CORPUS_DIR = DATA_ROOT / 'corpus'

RAWPDF_DIR         = PROJECT_ROOT / 'data' / 'rawpdf'
RAW_PDF_DIR        = RAWPDF_DIR / 'raw_pdfs'
OCR_OUTPUT_DIR     = RAWPDF_DIR / 'ocr_processed'
COMPARE_OUTPUT_DIR = RAWPDF_DIR / 'comparison'

RAW_PDF_DIR.mkdir(parents=True, exist_ok=True)
OCR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COMPARE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT    :', PROJECT_ROOT)
print('DATA_ROOT       :', DATA_ROOT)
print('RAWPDF_DIR      :', RAWPDF_DIR)

assert DATA_ROOT.exists(),       f'DATA_ROOT not found: {DATA_ROOT}'
assert PDF_URLS_PATH.exists(),   f'PDF_URLS_PATH not found: {PDF_URLS_PATH}'
assert JSON_CORPUS_DIR.exists(), f'JSON_CORPUS_DIR not found: {JSON_CORPUS_DIR}'
print('路径检查通过 ✓')

## 1. 文本清洗辅助函数

OCR 结果通常噪声更多，因此仍然需要和 JSON 流程一致的清洗逻辑。

In [ ]:
# 注意：使用 r'' raw string，反斜杠只写一次
REFERENCE_HEADING_PATTERN = re.compile(
    r'^(references|bibliography|acknowledg(e)?ments?|appendix)$', re.IGNORECASE
)
ARXIV_NOISE_PATTERN    = re.compile(r'arxiv:\s*\d{4}\.\d{4,5}(v\d+)?', re.IGNORECASE)
PAGE_NUMBER_PATTERN    = re.compile(r'^\s*\d+\s*$')
WHITESPACE_PATTERN     = re.compile(r'[ \t]+')
MULTI_NEWLINE_PATTERN  = re.compile(r'\n{3,}')
INLINE_NEWLINE_PATTERN = re.compile(r'(?<!\n)\n(?!\n)')
HYPHEN_BREAK_PATTERN   = re.compile(r'(\w+)-\n(\w+)')


def normalize_whitespace(text: str) -> str:
    text = text.replace('\xa0', ' ')
    text = WHITESPACE_PATTERN.sub(' ', text)
    text = MULTI_NEWLINE_PATTERN.sub('\n\n', text)
    return text.strip()


def fix_hyphenation(text: str) -> str:
    return HYPHEN_BREAK_PATTERN.sub(r'\1\2', text)


def merge_inline_newlines(text: str) -> str:
    return INLINE_NEWLINE_PATTERN.sub(' ', text)


def remove_noise_lines(text: str) -> str:
    cleaned_lines = []
    for line in text.splitlines():
        stripped = line.strip()
        if PAGE_NUMBER_PATTERN.fullmatch(stripped):
            continue
        if ARXIV_NOISE_PATTERN.search(stripped):
            continue
        cleaned_lines.append(line)
    return '\n'.join(cleaned_lines)


def clean_text(text: str, merge_lines: bool = False) -> str:
    text = fix_hyphenation(text)
    text = remove_noise_lines(text)
    if merge_lines:
        text = merge_inline_newlines(text)
    text = normalize_whitespace(text)
    return text


print('文本清洗函数定义完成 ✓')

## 2. 读取元数据并选择 50 篇论文

In [ ]:
def load_json(path: Path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)


pdf_urls = load_json(PDF_URLS_PATH)
paper_ids = list(pdf_urls.keys())[:50]

print(f'选择了 {len(paper_ids)} 篇论文')
print('示例 IDs:', paper_ids[:5])

## 3. 下载 Raw PDF

下载结果保存到：`project/data/processed/raw_pdfs/`。

In [ ]:
def download_pdf(paper_id: str, url: str) -> Path:
    """下载 PDF，若已存在则跳过。"""
    out_path = RAW_PDF_DIR / f'{paper_id}.pdf'
    if out_path.exists() and out_path.stat().st_size > 0:
        return out_path
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    out_path.write_bytes(response.content)
    return out_path


downloaded_paths = []
for paper_id in tqdm(paper_ids, desc='下载 PDF'):
    pdf_path = download_pdf(paper_id, pdf_urls[paper_id])
    downloaded_paths.append(pdf_path)

print(f'\n已下载或复用 {len(downloaded_paths)} 个 PDF 文件')
if downloaded_paths:
    print(f'示例路径: {downloaded_paths[0]}')

## 4. OCR / Markdown 提取函数

当前默认使用 `pymupdf4llm.to_markdown(...)`。

后续若切换为 PaddleOCR / PP-Structure，只需替换本单元中的提取逻辑。

In [ ]:
def extract_markdown_with_paddle(pdf_path: Path) -> str:
    """使用 pymupdf4llm 提取 PDF 为 markdown 格式。"""
    import pymupdf4llm

    md_text = pymupdf4llm.to_markdown(
        str(pdf_path),
        page_chunks=False,
        write_images=False,
        force_text=False,
    )
    return md_text


print('OCR 提取函数定义完成 ✓')

## 5. 将 Markdown 切分为 Sections

这里利用 markdown 标题（`#`, `##`, `###` 等）作为结构切分依据。

In [ ]:
def markdown_to_sections(md_text: str) -> list[dict]:
    """按 markdown heading 切分为 sections。"""
    heading_pattern = re.compile(r'^(#{1,6})\s+(.+)$', re.MULTILINE)
    matches = list(heading_pattern.finditer(md_text))

    if not matches:
        cleaned = clean_text(md_text, merge_lines=False)
        return [{'heading': 'full_document', 'text': cleaned}] if cleaned else []

    sections = []
    for idx, match in enumerate(matches):
        heading = match.group(2).strip()
        start = match.end()
        end = matches[idx + 1].start() if idx + 1 < len(matches) else len(md_text)
        body = md_text[start:end].strip()
        cleaned = clean_text(body, merge_lines=False)
        if len(cleaned) < 50:
            continue
        sections.append({'heading': heading, 'text': cleaned})
    return sections


print('Markdown 切分函数定义完成 ✓')

## 6. 将 Raw PDF 解析为 OCR JSON

这里加入缓存逻辑：如果 OCR 结果已经存在，则直接复用，避免重复解析。

In [ ]:
def parse_pdf_to_json(pdf_path: Path, paper_id: str) -> dict:
    """解析 PDF 为 JSON，带缓存。"""
    out_path = OCR_OUTPUT_DIR / f'{paper_id}.json'
    if out_path.exists():
        with open(out_path, 'r', encoding='utf-8') as f:
            return json.load(f)

    markdown = extract_markdown_with_paddle(pdf_path)
    sections = markdown_to_sections(markdown)
    parsed = {
        'id': paper_id,
        'source': 'raw_pdf_pymupdf4llm_markdown',
        'markdown': markdown,
        'sections': sections,
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(parsed, f, ensure_ascii=False, indent=2)
    return parsed


# 测试单个文件
if downloaded_paths:
    sample_doc = parse_pdf_to_json(downloaded_paths[0], paper_ids[0])
    print('示例文档 keys:', list(sample_doc.keys()))
    if sample_doc.get('sections'):
        print('第一个 section heading:', sample_doc['sections'][0]['heading'])
else:
    print('无可用 PDF')

## 7. 批量解析 50 篇 Raw PDF

In [ ]:
ocr_docs = []
for pdf_path, paper_id in tqdm(list(zip(downloaded_paths, paper_ids)), desc='OCR 解析'):
    ocr_doc = parse_pdf_to_json(pdf_path, paper_id)
    ocr_docs.append(ocr_doc)

print(f'\n已解析 {len(ocr_docs)} 个 OCR 文档')

## 8. 与官方 JSON 结果对比

这里先做基础统计对比：
- section 数量；
- 总字符数；
- OCR / JSON 的字符覆盖率。

对比结果保存到：`project/data/processed/comparison/`。

In [ ]:
def compare_with_reference(paper_id: str, ocr_doc: dict) -> dict:
    """对比 OCR 结果与官方 JSON corpus。"""
    ref_path = JSON_CORPUS_DIR / f'{paper_id}.json'
    if not ref_path.exists():
        return {'paper_id': paper_id, 'status': 'missing_reference'}

    ref_doc = load_json(ref_path)
    ref_sections = [
        clean_text(section.get('text', ''), merge_lines=True)
        for section in ref_doc.get('sections', [])
    ]
    ref_sections = [text for text in ref_sections if len(text) >= 50]

    ocr_sections = [
        section['text']
        for section in ocr_doc.get('sections', [])
        if len(section.get('text', '')) >= 50
    ]

    ref_chars = sum(len(text) for text in ref_sections)
    ocr_chars = sum(len(text) for text in ocr_sections)
    ratio = round(ocr_chars / ref_chars, 4) if ref_chars else 0.0

    comparison = {
        'paper_id': paper_id,
        'reference_sections': len(ref_sections),
        'ocr_sections': len(ocr_sections),
        'reference_chars': ref_chars,
        'ocr_chars': ocr_chars,
        'char_coverage_ratio': ratio,
    }

    out_path = COMPARE_OUTPUT_DIR / f'{paper_id}_comparison.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(comparison, f, ensure_ascii=False, indent=2)

    return comparison


all_results = []
for paper_id, ocr_doc in zip(paper_ids, ocr_docs):
    result = compare_with_reference(paper_id, ocr_doc)
    all_results.append(result)

summary_path = COMPARE_OUTPUT_DIR / 'summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(all_results, f, ensure_ascii=False, indent=2)

print(f'\n[OK] 对比结果已保存: {summary_path}')
if all_results:
    print('示例对比结果:', all_results[0])